# Улучшенная ML модель: Предсказание цен автомобилей

## Ключевые улучшения:
1. **Ансамбль моделей** (CatBoost + XGBoost + LightGBM) вместо одной
2. **Cross-validation** для более честной оценки
3. **Feature engineering v2.0**:
   - Полиномиальные фичи
   - Взаимодействия между брендом, объемом двигателя и возрастом
   - Целевое кодирование для категорий
4. **Hyperparameter tuning** вместо стандартных параметров
5. **Weighted average** предсказаний моделей
6. **Калибровка** для лучшей точности интервалов
7. **Регуляризация** для избежания переобучения

In [34]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.preprocessing import TargetEncoder

import catboost as cb
import xgboost as xgb
import lightgbm as lgb

CURRENT_YEAR = 2026
DATA_PATH = "cars_ml_10k.csv"
RANDOM_STATE = 42

df_raw = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df_raw.shape}")
print(f"\nColumns: {df_raw.columns.tolist()}")
print(f"\nFirst rows:")
df_raw.head()

Dataset shape: (10542, 14)

Columns: ['brand', 'model', 'year', 'price', 'city', 'mileage_km', 'body_type', 'engine_volume_l', 'fuel_type', 'transmission', 'drive_type', 'steering_wheel', 'color', 'generation']

First rows:


,brand,model,year,price,city,mileage_km,body_type,engine_volume_l,fuel_type,transmission,drive_type,steering_wheel,color,generation
0,Toyota,Highlander Luxe,2017,17590000,Астана,230186.0,Кроссовер,3.5,бензин,Автомат,Полный привод,Слева,черный,U5
1,Rox,01 VIP,2026,30490000,Алматы,NaN,Внедорожник,1.5,гибрид,Автомат,Полный привод,Слева,NaN,unknown
2,Hyundai,Elantra,2020,7700000,Алматы,183426.0,Седан,2.0,бензин,Автомат,Передний привод,Слева,серый металлик,AD/ADA
3,Kia,Sportage,2014,7000000,Петропавловск,114500.0,Кроссовер,2.0,бензин,Механика,Передний привод,Слева,серый,SL
4,Toyota,Land Cruiser Prado,2014,14999999,Актобе,NaN,Внедорожник,2.7,бензин,Автомат,Полный привод,Слева,NaN,J150


## 1. Data Cleaning & EDA

In [35]:
print("Пропуски в данных:")
print(df_raw.isnull().sum())
print(f"\nСтатистика цен: ₸{df_raw['price'].min():,.0f} - ₸{df_raw['price'].max():,.0f}")
print(f"Лет: {df_raw['year'].min()} - {df_raw['year'].max()}")
print(f"Топ брендов: {df_raw['brand'].value_counts().head(10).to_dict()}")

Пропуски в данных:
brand                 0
model                 2
year                  0
price                 0
city                  0
mileage_km         3045
body_type             0
engine_volume_l     175
fuel_type             3
transmission          1
drive_type            1
steering_wheel        1
color              3729
generation            0
dtype: int64

Статистика цен: ₸60,000 - ₸223,000,000
Лет: 1964 - 2026
Топ брендов: {'Toyota': 1847, 'Hyundai': 1687, 'BMW': 1521, 'ВАЗ': 570, 'Kia': 517, 'Mercedes-Benz': 491, 'Lexus': 313, 'Chevrolet': 261, 'Volkswagen': 252, 'Nissan': 240}


model
Camry               664
X5                  328
Tucson              248
Accent              227
Elantra             226
                   ... 
Elantra Elegance      1
Casper                1
Santamo               1
Sonata Luxe           1
Trajet                1
Name: count, Length: 1294, dtype: int64

## 2. Advanced Data Processing

In [46]:
def process_data_v2(df, is_train=True, target_encoders=None, label_encoders=None):
    """Улучшенная обработка данных с целевым кодированием"""
    df = df.copy()
    counts = df['model'].value_counts()
    df= df[df['model'].isin(counts[counts >= 100].index)]
    # 1. Удаляем явные выбросы только на train.
    # На predict нельзя фильтровать строку, иначе manual_car с price=0 исчезнет.
    if is_train:
        df = df[df['price'] > 100_000]
        df = df[df['price'] < 200_000_000]
        df = df[df['year'] >= 1990]
        df = df[df['engine_volume_l'] > 0]
    else:
        if 'price' not in df.columns:
            df['price'] = 0

    
    # 2. Целевая переменная (log-трансформация)
    y = np.log1p(df['price']) if is_train else None
    
    # 3. Численные фичи
    df['car_age'] = CURRENT_YEAR - df['year']
    df['mileage_km'] = df['mileage_km'].fillna(df['mileage_km'].median())
    df['mileage_per_year'] = df['mileage_km'] / (df['car_age'] + 1)
    
    # 4. Новые фичи из engine_volume и car_age
    df['engine_volume_sq'] = df['engine_volume_l'] ** 2
    df['age_times_volume'] = df['car_age'] * df['engine_volume_l']
    df['age_sq'] = df['car_age'] ** 2
    
    # 5. Flags
    df['mileage_missing'] = df['mileage_km'].isna().astype(int)
    if 'generation_code' not in df.columns:
        df['generation_code'] = 'unknown'
    df['generation_missing'] = df['generation_code'].isna().astype(int)
    df['is_new'] = ((CURRENT_YEAR - df['year']) <= 1).astype(int)
    df['is_luxury'] = df['brand'].isin(['Porsche', 'BMW', 'Mercedes', 'Audi', 'Lexus']).astype(int)
    
    # 6. Generation parsing
    def extract_gen_number(gen_str):
        if pd.isna(gen_str):
            return 0
        gen_str = str(gen_str)
        match = re.search(r'(\d+)\s*поколение', gen_str)
        return int(match.group(1)) if match else 0
    
    df['generation_number'] = df['generation'].apply(extract_gen_number)
    df['has_gen_number'] = (df['generation_number'] > 0).astype(int)
    
    # 7. Категориальные фичи: группировка редких категорий
    cat_columns = [
        'brand', 'model', 'city', 'body_type', 'fuel_type',
        'transmission', 'drive_type', 'steering_wheel', 'color',
        'generation_code'
    ]
    
    for col in cat_columns:
        if col in df.columns:
            df[col] = df[col].fillna('unknown')
            
            if is_train:
                # Группируем редкие категории (< 10 упоминаний)
                freq = df[col].value_counts()
                rare = freq[freq < 10].index
                df[col] = df[col].replace(rare, 'other')
    
    # 8. Целевое кодирование брендов и моделей
    if is_train:
        target_encoders = {}
        for col in ['brand', 'model', 'body_type']:
            encoder = TargetEncoder(smooth=1.0)
            df[f'{col}_target_encoded'] = encoder.fit_transform(
                df[[col]].astype(str), y
            )
            target_encoders[col] = encoder
    else:
        for col in ['brand', 'model', 'body_type']:
            if col in target_encoders:
                df[f'{col}_target_encoded'] = target_encoders[col].transform(
                    df[[col]].astype(str)
                )
    
    # 9. Label encoding для категорий.
    # В predict unseen categories заменяем на 'unknown', чтобы не было ошибки.
    if is_train:
        label_encoders = {}
        for col in cat_columns:
            le = LabelEncoder()
            df[col] = df[col].astype(str).fillna('unknown')
            if 'unknown' not in set(df[col].unique()):
                df.loc[df.index[:1], col] = 'unknown'
            df[f'{col}_le'] = le.fit_transform(df[col].astype(str))
            label_encoders[col] = le
    else:
        for col in cat_columns:
            if col in label_encoders:
                le = label_encoders[col]
                if col not in df.columns:
                    df[col] = 'unknown'
                values = df[col].astype(str).fillna('unknown')
                known_values = set(le.classes_)
                values = values.apply(lambda x: x if x in known_values else 'unknown')
                if 'unknown' not in known_values:
                    le.classes_ = np.append(le.classes_, 'unknown')
                df[f'{col}_le'] = le.transform(values)
    
    # 10. Select features
    numeric_features = [
        'year', 'mileage_km', 'engine_volume_l', 'car_age', 'mileage_per_year',
        'generation_number', 'engine_volume_sq', 'age_times_volume', 'age_sq'
    ]
    
    flag_features = [
        'mileage_missing', 'generation_missing', 'is_new', 'is_luxury', 'has_gen_number'
    ]
    
    cat_with_encoding = [
        f'{col}_le' for col in cat_columns
    ] + [
        f'{col}_target_encoded' for col in ['brand', 'model', 'body_type']
    ]
    
    all_features = numeric_features + flag_features + cat_with_encoding
    all_features = [f for f in all_features if f in df.columns]
    
    X = df[all_features].copy()
    
    # Заполняем NaN (на случай целевого кодирования при predict)
    X = X.fillna(X.mean(numeric_only=True))
    X = X.fillna(-1)
    
    return X, y if is_train else None, all_features, target_encoders, label_encoders


X, y, features, target_enc, label_enc = process_data_v2(df_raw, is_train=True)
print(f"Features shape: {X.shape}")
print(f"Features: {features[:5]}... (всего {len(features)})")

Features shape: (3050, 27)
Features: ['year', 'mileage_km', 'engine_volume_l', 'car_age', 'mileage_per_year']... (всего 27)


## 3. Train/Val Split

In [47]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape}, Val: {X_val.shape}")
print(f"y_train shape: {y_train.shape}")

Train: (2440, 27), Val: (610, 27)
y_train shape: (2440,)


## 4. CatBoost (Tuned)

In [53]:
cat_model = cb.CatBoostRegressor(
    iterations=5000,
    learning_rate=0.02,
    max_depth=8,
    l2_leaf_reg=3.0,
    subsample=0.8,
    random_strength=0.15,
    bagging_temperature=0.8,
    od_type='Iter',
    od_wait=500,
    verbose=False,
    random_state=RANDOM_STATE,
    thread_count=-1,
    loss_function='MAE'
)

cat_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    use_best_model=True,
    verbose=False
)

cat_pred = cat_model.predict(X_val)
cat_mae = mean_absolute_error(y_val, cat_pred)
cat_rmse = np.sqrt(mean_squared_error(y_val, cat_pred))
cat_r2 = r2_score(y_val, cat_pred)

print(f"CatBoost (log scale):")
print(f"  MAE:  {cat_mae:.6f}")
print(f"  RMSE: {cat_rmse:.6f}")
print(f"  R2:   {cat_r2:.6f}")

CatBoost (log scale):
  MAE:  0.134501
  RMSE: 0.218561
  R2:   0.898925


## 9. Feature Importance (CatBoost)

In [54]:
importance = pd.DataFrame({
    "feature": features,
    "importance": cat_model.get_feature_importance()
}).sort_values("importance", ascending=False)

print("\n=== Топ 15 важных фич ===")
print(importance.head(15).to_string(index=False))


=== Топ 15 важных фич ===
                 feature  importance
    model_target_encoded   32.804189
                    year    9.474543
                 car_age    8.218074
                  age_sq    7.953149
    brand_target_encoded    5.725986
              mileage_km    5.308493
                model_le    5.162265
                 city_le    3.295975
         engine_volume_l    2.986074
                brand_le    2.730914
                color_le    2.685321
        age_times_volume    2.575268
        engine_volume_sq    2.542773
body_type_target_encoded    2.377482
        mileage_per_year    2.315768


## 10. Prediction Function

In [51]:
def predict_car_price(car_data):
    """
    Предсказываем цену машины с доверительным интервалом.
    
    car_data: dict с параметрами машины
    """
    car_df = pd.DataFrame([car_data])
    
    # Обработка данных
    X_car, _, _, _, _ = process_data_v2(
        car_df, 
        is_train=False,
        target_encoders=target_enc,
        label_encoders=label_enc
    )
    
    X_car = X_car.reindex(columns=features, fill_value=-1)
    X_car = X_car.fillna(X_car.mean(numeric_only=True).to_dict())
    X_car = X_car.fillna(-1)
    
    # Предсказания
    cat_pred_log = cat_model.predict(X_car)[0]

    
    # Ансамбль
    ensemble_pred_log =  cat_pred_log
    
    # Обратное преобразование
    pred_price = np.expm1(ensemble_pred_log)
    
    # Доверительный интервал (более узкий благодаря ансамблю)
    # Используем остатки из валидации для оценки ошибки
    std_error = np.std(pred_price)
    
    # Интервалы: mean ± 1.96*std (95% confidence)
    low = pred_price * 0.85
    high = pred_price * 1.15
        
    return pred_price, low, high, {
        'cat': np.expm1(cat_pred_log),
        
    }


# Тестовый пример
test_car = {
    "brand": "Toyota",
    "model": "Camry",
    "year": 2017,
    "city": "Алматы",
    "mileage_km": 150000,
    "body_type": "Седан",
    "engine_volume_l": 2.5,
    "fuel_type": "бензин",
    "transmission": "Автомат",
    "drive_type": "Передний привод",
    "steering_wheel": "Слева",
    "color": "Серый",
    "generation": "XV50",
}

price, low, high, individual_preds = predict_car_price(test_car)

print(f"\n=== ТЕСТОВЫЙ ПРИМЕР ===")
print(f"Машина: {test_car['brand']} {test_car['model']} {test_car['year']}")
print(f"Параметры: {test_car['engine_volume_l']}L, {test_car['mileage_km']:,} км")
print(f"\n╔══════════════════════════════════╗")
print(f"║ Предсказанная цена: ₸{price:>15,.0f} ║")
print(f"║ Диапазон: ₸{low:>13,.0f} - {high:>13,.0f} ║")
print(f"╚══════════════════════════════════╝")
print(f"\nПредсказания отдельных моделей:")
print(f"  CatBoost: ₸{individual_preds['cat']:>15,.0f}")


ValueError: Found array with 0 sample(s) (shape=(0,)) while a minimum of 1 is required.

## 11. Model Comparison

In [55]:
df_raw.head()

,brand,model,year,price,city,mileage_km,body_type,engine_volume_l,fuel_type,transmission,drive_type,steering_wheel,color,generation
0,Toyota,Highlander Luxe,2017,17590000,Астана,230186.0,Кроссовер,3.5,бензин,Автомат,Полный привод,Слева,черный,U5
1,Rox,01 VIP,2026,30490000,Алматы,NaN,Внедорожник,1.5,гибрид,Автомат,Полный привод,Слева,NaN,unknown
2,Hyundai,Elantra,2020,7700000,Алматы,183426.0,Седан,2.0,бензин,Автомат,Передний привод,Слева,серый металлик,AD/ADA
3,Kia,Sportage,2014,7000000,Петропавловск,114500.0,Кроссовер,2.0,бензин,Механика,Передний привод,Слева,серый,SL
4,Toyota,Land Cruiser Prado,2014,14999999,Актобе,NaN,Внедорожник,2.7,бензин,Автомат,Полный привод,Слева,NaN,J150
